In [0]:
select *
from cmpa_insights_internal_schema.tier1_hcos_detailed limit 5

In [0]:
select *
from cmpa_insights_internal_schema.reference_file_hcos limit 5

In [0]:
%python
%pip install rapidfuzz
dbutils.library.restartPython()

In [0]:
%python
       
"""
HCO Matching Script
-------------------
Matches reference_file_hcos against child HCOs in tier1_hcos_detailed using:
  1. ZIP-based pre-filtering (exact match)
  2. Fuzzy name + address similarity (rapidfuzz)
  3. Threshold-based confirmation

Output: reference HCO rows with confirmed matches tagged with matched child info
        and parent (primary) info from tier1_hcos_detailed.

Requirements:
    pip install rapidfuzz pandas databricks-connect
    (or run directly in a Databricks notebook — just remove the spark session init)
"""

import pandas as pd
from rapidfuzz import fuzz
from databricks.connect import DatabricksSession  # Remove if running inside Databricks notebook

# ─────────────────────────────────────────────
# 1. SPARK SESSION
#    If running inside a Databricks notebook, replace with: spark = spark
# ─────────────────────────────────────────────
spark = DatabricksSession.builder.getOrCreate()  # Remove/replace if inside Databricks notebook


# ─────────────────────────────────────────────
# 2. CONFIG
# ─────────────────────────────────────────────
SIMILARITY_THRESHOLD = 85          # Minimum score (0–100) to confirm a match
NAME_WEIGHT          = 0.6         # Weight for name similarity
ADDRESS_WEIGHT       = 0.4         # Weight for address similarity


# ─────────────────────────────────────────────
# 3. LOAD DATA FROM DATABRICKS
# ─────────────────────────────────────────────
print("Loading tables from Databricks...")

tier1_df = spark.table("cmpa_insights_internal_schema.tier1_hcos_detailed").toPandas()
ref_df   = spark.table("cmpa_insights_internal_schema.reference_file_hcos").toPandas()

print(f"  tier1_hcos_detailed  : {len(tier1_df):,} rows")
print(f"  reference_file_hcos  : {len(ref_df):,} rows")


# ─────────────────────────────────────────────
# 4. NORMALIZE HELPER
# ─────────────────────────────────────────────
def normalize(text: str) -> str:
    """Lowercase, strip extra whitespace."""
    if pd.isna(text):
        return ""
    return str(text).lower().strip()


def normalize_zip(zip_val) -> str:
    """Keep only first 5 digits for ZIP matching."""
    if pd.isna(zip_val):
        return ""
    return str(zip_val).strip().split("-")[0].zfill(5)


# ─────────────────────────────────────────────
# 5. NORMALIZE COLUMNS
# ─────────────────────────────────────────────
# Reference file
ref_df["hco_name_norm"]    = ref_df["hco_name"].apply(normalize)
ref_df["hco_address_norm"] = ref_df["hco_address"].apply(normalize)
ref_df["hco_zip_norm"]     = ref_df["hco_zip"].apply(normalize_zip)

# Tier1 child columns
tier1_df["child_name_norm"]    = tier1_df["child_name"].apply(normalize)
tier1_df["child_address_norm"] = tier1_df["child_address"].apply(normalize)
tier1_df["child_zip_norm"]     = tier1_df["child_zip"].apply(normalize_zip)


# ─────────────────────────────────────────────
# 6. BUILD ZIP → CHILD ROWS INDEX
#    Groups tier1 child rows by ZIP for fast lookup
# ─────────────────────────────────────────────
print("Building ZIP index...")

zip_index: dict[str, list[dict]] = {}
for row in tier1_df.itertuples(index=False):
    z = row.child_zip_norm
    if z not in zip_index:
        zip_index[z] = []
    zip_index[z].append({
        "child_name"      : row.child_name,
        "child_address"   : row.child_address,
        "child_zip"       : row.child_zip,
        "primary_name"    : row.primary_name,
        "primary_address" : row.primary_address,
        "primary_zip"     : row.primary_zip,
        "name_norm"       : row.child_name_norm,
        "address_norm"    : row.child_address_norm,
    })

print(f"  Unique ZIPs in tier1 children : {len(zip_index):,}")


# ─────────────────────────────────────────────
# 7. MATCHING FUNCTION
# ─────────────────────────────────────────────
def find_best_match(ref_name_norm: str,
                    ref_address_norm: str,
                    ref_zip_norm: str) -> dict | None:
    """
    Returns the best matching child record (dict) if score >= threshold, else None.
    Uses weighted combination of token_set_ratio for name and address.
    """
    candidates = zip_index.get(ref_zip_norm)
    if not candidates:
        return None

    best_score  = -1.0
    best_record = None

    for cand in candidates:
        name_score    = fuzz.token_set_ratio(ref_name_norm,    cand["name_norm"])
        address_score = fuzz.token_set_ratio(ref_address_norm, cand["address_norm"])
        combined      = NAME_WEIGHT * name_score + ADDRESS_WEIGHT * address_score

        if combined > best_score:
            best_score  = combined
            best_record = cand

    if best_score >= SIMILARITY_THRESHOLD:
        return {**best_record, "_match_score": round(best_score, 2)}
    return None


# ─────────────────────────────────────────────
# 8. RUN MATCHING ACROSS ALL REFERENCE HCOs
# ─────────────────────────────────────────────
print("Running matching...")

matched_rows = []
unmatched_count = 0

for idx, row in ref_df.iterrows():
    result = find_best_match(
        row["hco_name_norm"],
        row["hco_address_norm"],
        row["hco_zip_norm"],
    )

    if result:
        matched_rows.append({
            # Original reference columns
            "hco_name"              : row["hco_name"],
            "hco_address"           : row["hco_address"],
            "hco_zip"               : row["hco_zip"],
            # Matched child info
            "matched_child_name"    : result["child_name"],
            "matched_child_address" : result["child_address"],
            "matched_child_zip"     : result["child_zip"],
            # Parent (primary) info
            "primary_name"          : result["primary_name"],
            "primary_address"       : result["primary_address"],
            "primary_zip"           : result["primary_zip"],
            # Confidence score
            "match_score"           : result["_match_score"],
        })
    else:
        unmatched_count += 1

print(f"\n  Matched   : {len(matched_rows):,}")
print(f"  Unmatched : {unmatched_count:,}")


# ─────────────────────────────────────────────
# 9. BUILD OUTPUT DATAFRAME
# ─────────────────────────────────────────────
output_df = pd.DataFrame(matched_rows, columns=[
    "hco_name", "hco_address", "hco_zip",
    "matched_child_name", "matched_child_address", "matched_child_zip",
    "primary_name", "primary_address", "primary_zip",
    "match_score",
])

# Sort by match score descending for easy review
output_df = output_df.sort_values("match_score", ascending=False).reset_index(drop=True)

print("\nSample output (top 5):")
print(output_df.head().to_string(index=False))


# ─────────────────────────────────────────────
# 10. SAVE OUTPUT
#     Option A: Save as CSV locally
#     Option B: Write back to Databricks as a table (uncomment)
# ─────────────────────────────────────────────

# --- Option A: CSV ---
output_path = "hco_matching_results.csv"
output_df.to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path}")

# --- Option B: Write to Databricks table (uncomment to use) ---
# spark_output = spark.createDataFrame(output_df)
# spark_output.write.mode("overwrite").saveAsTable("cmpa_insights_internal_schema.reference_hcos_with_tier1_parent")
# print("Results written to Databricks table: reference_hcos_with_tier1_parent")

In [0]:
%python
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
SIMILARITY_THRESHOLD = 85   # Minimum score (0–100) to confirm a match
NAME_WEIGHT          = 0.6  # Weight for name similarity
ADDRESS_WEIGHT       = 0.4  # Weight for address similarity

# ─────────────────────────────────────────────
# INSTALL DEPENDENCY (run once, then restart if needed)
# ─────────────────────────────────────────────
# %pip install rapidfuzz

# ─────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────
import pandas as pd
from rapidfuzz import fuzz

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
tier1_df = spark.table("cmpa_insights_internal_schema.tier1_hcos_detailed").toPandas()
ref_df   = spark.table("cmpa_insights_internal_schema.reference_file_hcos").toPandas()

print(f"tier1_hcos_detailed  : {len(tier1_df):,} rows")
print(f"reference_file_hcos  : {len(ref_df):,} rows")

# ─────────────────────────────────────────────
# NORMALIZE HELPERS
# ─────────────────────────────────────────────
def normalize(text) -> str:
    if pd.isna(text):
        return ""
    return str(text).lower().strip()

def normalize_zip(zip_val) -> str:
    if pd.isna(zip_val):
        return ""
    return str(zip_val).strip().split("-")[0].zfill(5)

# ─────────────────────────────────────────────
# NORMALIZE COLUMNS  (no leading underscores — itertuples mangles those)
# ─────────────────────────────────────────────
ref_df["hco_name_norm"]    = ref_df["hco_name"].apply(normalize)
ref_df["hco_address_norm"] = ref_df["hco_address"].apply(normalize)
ref_df["hco_zip_norm"]     = ref_df["hco_zip"].apply(normalize_zip)

tier1_df["child_name_norm"]    = tier1_df["child_name"].apply(normalize)
tier1_df["child_address_norm"] = tier1_df["child_address"].apply(normalize)
tier1_df["child_zip_norm"]     = tier1_df["child_zip"].apply(normalize_zip)

# ─────────────────────────────────────────────
# BUILD ZIP → CHILD ROWS INDEX
# ─────────────────────────────────────────────
zip_index = {}
for row in tier1_df.itertuples(index=False):
    z = row.child_zip_norm
    if z not in zip_index:
        zip_index[z] = []
    zip_index[z].append({
        "child_name"     : row.child_name,
        "child_address"  : row.child_address,
        "child_zip"      : row.child_zip,
        "primary_name"   : row.primary_name,
        "primary_address": row.primary_address,
        "primary_zip"    : row.primary_zip,
        "name_norm"      : row.child_name_norm,
        "address_norm"   : row.child_address_norm,
    })

print(f"Unique ZIPs in tier1 children : {len(zip_index):,}")

# ─────────────────────────────────────────────
# MATCHING FUNCTION
# ─────────────────────────────────────────────
def find_best_match(ref_name_norm, ref_address_norm, ref_zip_norm):
    candidates = zip_index.get(ref_zip_norm)
    if not candidates:
        return None

    best_score  = -1.0
    best_record = None

    for cand in candidates:
        name_score    = fuzz.token_set_ratio(ref_name_norm,    cand["name_norm"])
        address_score = fuzz.token_set_ratio(ref_address_norm, cand["address_norm"])
        combined      = NAME_WEIGHT * name_score + ADDRESS_WEIGHT * address_score

        if combined > best_score:
            best_score  = combined
            best_record = cand

    if best_score >= SIMILARITY_THRESHOLD:
        return {**best_record, "match_score": round(best_score, 2)}
    return None

# ─────────────────────────────────────────────
# RUN MATCHING
# ─────────────────────────────────────────────
matched_rows = []

for _, row in ref_df.iterrows():
    result = find_best_match(row["hco_name_norm"], row["hco_address_norm"], row["hco_zip_norm"])
    if result:
        matched_rows.append({
            "hco_name"             : row["hco_name"],
            "hco_address"          : row["hco_address"],
            "hco_zip"              : row["hco_zip"],
            "matched_child_name"   : result["child_name"],
            "matched_child_address": result["child_address"],
            "matched_child_zip"    : result["child_zip"],
            "primary_name"         : result["primary_name"],
            "primary_address"      : result["primary_address"],
            "primary_zip"          : result["primary_zip"],
            "match_score"          : result["match_score"],
        })

# ─────────────────────────────────────────────
# OUTPUT DATAFRAME
# ─────────────────────────────────────────────
output_df = pd.DataFrame(matched_rows).sort_values("match_score", ascending=False).reset_index(drop=True)

print(f"\nMatched   : {len(output_df):,}")
print(f"Unmatched : {len(ref_df) - len(output_df):,}")

display(output_df)

In [0]:
%python
output_df.display()